In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/loan_approval_dataset.csv"
OUT_PATH = "../data/processed_loans.csv"

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.head()


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [2]:
if "loan_id" in df.columns:
    df = df.drop(columns=["loan_id"])

y_raw = df["loan_status"].astype(str).str.strip().str.lower()
df["target"] = y_raw.map({"rejected": 0, "approved": 1})

bad = df[df["target"].isna()]["loan_status"].unique()
print("Unmapped loan_status values:", bad)

df = df.dropna(subset=["target"]).copy()
df["target"] = df["target"].astype(int)

df["loan_status"].value_counts()


Unmapped loan_status values: []


loan_status
Approved    2656
Rejected    1613
Name: count, dtype: int64

In [3]:
income = df["income_annum"].replace(0, np.nan)
term = df["loan_term"].replace(0, np.nan)

# affordability ratio (loan_amount not used directly later)
df["emi_income_ratio"] = (df["loan_amount"] / term) / income
df["emi_income_ratio"] = df["emi_income_ratio"].replace([np.inf, -np.inf], np.nan).fillna(1.0)

# total assets (combine 4 into 1)
df["asset_total"] = (
    df["residential_assets_value"]
    + df["commercial_assets_value"]
    + df["luxury_assets_value"]
    + df["bank_asset_value"]
)

df["is_not_graduate"] = (df["education"].astype(str).str.strip().str.lower() == "not graduate").astype(int)
df["is_self_employed"] = (df["self_employed"].astype(str).str.strip().str.lower() == "yes").astype(int)

# reduce cibil dominance using banding
df["cibil_band"] = pd.cut(
    df["cibil_score"],
    bins=[0, 550, 650, 750, 900],
    labels=[0, 1, 2, 3],
    include_lowest=True
).astype(int)

df[["emi_income_ratio", "asset_total", "cibil_band", "target"]].head()


,emi_income_ratio,asset_total,cibil_band,target
0,0.259549,50700000,3,1
1,0.371951,17000000,0,0
2,0.163187,57700000,0,0
3,0.467988,52700000,0,0
4,0.123469,55000000,0,0


In [4]:
# Keep only necessary model features
feature_cols = [
    "income_annum",
    "emi_income_ratio",
    "no_of_dependents",
    "is_not_graduate",
    "is_self_employed",
    "asset_total",
    "cibil_band"
]

processed = df[feature_cols + ["target"]].copy()

processed.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
processed.head()


Saved: ../data/processed_loans.csv


,income_annum,emi_income_ratio,no_of_dependents,is_not_graduate,is_self_employed,asset_total,cibil_band,target
0,9600000,0.259549,2,0,0,50700000,3,1
1,4100000,0.371951,0,1,1,17000000,0,0
2,9100000,0.163187,3,0,0,57700000,0,0
3,8200000,0.467988,3,0,0,52700000,0,0
4,9800000,0.123469,5,1,1,55000000,0,0
